# map/ — Colab runner (Parametric UMAP / TensorFlow)

Chạy pipeline bản đồ trên Colab — `pumap2d` (Parametric UMAP cần TensorFlow; local mac py3.9/macOS26 crash).
Code clone từ GitHub; **data đặt sẵn trên Drive** `MyDrive/map_data/`; coords/reducer/HTML ghi thẳng về Drive.

## Đặt các folder này vào `MyDrive/map_data/` (1 lần)
```
MyDrive/map_data/
├── artifacts/      # item_vectors.npy, item_index.parquet, user_tower.pt, users_history.parquet, user_split.parquet
└── cleaned-data/   # details.csv
```
Notebook tự symlink các folder này vào path repo + `map/outputs/` → `MyDrive/map_data/outputs/`.
Kéo HTML ở `outputs/` về local mở browser (zoom/hover).

> map/ chỉ còn pumap2d (2D). Sphere + genre-probe (`encode_genre.py`) đã bỏ → KHÔNG cần
> `checkpoints/best.pt` hay `train-data/` nữa.

In [ ]:
# 0) Mount Drive + clone code từ GitHub + cd vào repo
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

MAP_DATA = '/content/drive/MyDrive/map_data'
assert os.path.isdir(MAP_DATA), f'Khong thay {MAP_DATA} — tao folder + upload data (xem cell tren).'

REPO   = 'https://github.com/CrocodixD/anime-recommender.git'
BRANCH = 'main'      # ⚠ map/ + code mới nhất nằm ở main (anime_map đã cũ, ĐỪNG dùng)
CODE   = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !git clone -b {BRANCH} {REPO} {CODE}

%cd {CODE}

In [ ]:
# 1) Symlink data trên Drive vào đúng path repo (code dùng path tương đối từ ROOT repo)
import os

assert os.path.isdir(f'{CODE}/map'), (
    f'Khong thay {CODE}/map — folder map/ chua co trong clone.\n'
    'Push folder map/ len GitHub roi set BRANCH (cell 0) dung branch da push.')

os.makedirs(f'{MAP_DATA}/outputs', exist_ok=True)   # outputs/ ghi nguoc ve Drive
links = {
    f'{MAP_DATA}/artifacts':    f'{CODE}/artifacts',
    f'{MAP_DATA}/cleaned-data': f'{CODE}/cleaned-data',
    f'{MAP_DATA}/outputs':      f'{CODE}/map/outputs',
}
for src, dst in links.items():
    assert os.path.isdir(src), f'Thieu {src} tren Drive'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        raise SystemExit(f'{dst} da ton tai (khong phai symlink) — xoa tay roi chay lai.')
    os.symlink(src, dst)
    print('link', dst, '->', src)

In [ ]:
# 2) Cài lib + ÉP TensorFlow chạy CPU (GPU Blackwell cc12.0 chưa có CUDA kernel -> INVALID_PTX)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'    # áp dụng cho MỌI !python sau đó (pumap fit + transform)
!pip install -q umap-learn                   # TF/torch/plotly/sklearn Colab có sẵn
import tensorflow as tf, torch
print('tf', tf.__version__, '| torch', torch.__version__, '| GPU TF:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3) Sanity: file đầu vào đã link đủ chưa
import os
need = ['artifacts/item_vectors.npy','artifacts/item_index.parquet','artifacts/user_tower.pt',
        'artifacts/users_history.parquet','artifacts/user_split.parquet',
        'cleaned-data/details.csv']
miss = [f for f in need if not os.path.exists(f)]
print('THIEU:', miss) if miss else print('OK — đủ file đầu vào.')

In [ ]:
# 4) Base 1 lần (join vector + metadata genre)
!python map/build_base.py

In [ ]:
# 5) Fit projection — pumap2d (TF, CPU). Lưu coords + reducer (ParametricUMAP).
# Config CHỐT sau sweep: n_neighbors=50, min_dist=0.8 (spread mềm/tròn, lãnh thổ liền mạch — map/README.md).
print('===== project pumap2d (n_neighbors=50, min_dist=0.8) =====')
!python map/project.py --n-neighbors 50 --min-dist 0.8

In [ ]:
# 6) Cluster 128-d (để tô màu chéo mọi map)
!python map/cluster.py --algo kmeans --k 20

In [ ]:
# 7) Đặt user ("you are here") rồi render HTML pumap2d (color=cluster theo tên cụm).
import pandas as pd
s = pd.read_parquet('artifacts/user_split.parquet')
print('username mẫu:', [u for u in s['username'] if u[0].isalnum()][:5])

USERNAME = ''                       # <-- điền 1 username để có 'you are here' (để trống = chỉ vẽ cụm)

overlays = []
if USERNAME:
    !python map/encode_user.py "{USERNAME}"
    overlays.append(f'overlay_user_{USERNAME}')
ov = ' '.join(f'"{o}"' for o in overlays)
!python map/viz.py --method pumap2d --color cluster --cluster kmeans --overlay {ov} --suffix demo

In [ ]:
# 8) Xác nhận HTML đã nằm trên Drive (kéo về local mở browser)
!ls -lhS /content/drive/MyDrive/map_data/outputs/*.html